In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from google.colab import files
files.upload()  # ⬆️ Upload your kaggle.json file
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d mohammadhossein77/brain-tumors-dataset -p /content && unzip -q /content/brain-tumors-dataset.zip -d /content/brain_tumor_data
!kaggle datasets download -d gauravsrivastav2507/ehr-dataset -p /content && unzip -q /content/ehr-dataset.zip -d /content/ehr-data


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mohammadhossein77/brain-tumors-dataset
License(s): CC0-1.0
 85% 188M/221M [00:00<00:00, 1.91GB/s]
100% 221M/221M [00:00<00:00, 1.23GB/s]
Dataset URL: https://www.kaggle.com/datasets/gauravsrivastav2507/ehr-dataset
License(s): unknown
  0% 0.00/603k [00:00<?, ?B/s]
100% 603k/603k [00:00<00:00, 1.32GB/s]


In [3]:
import os
import shutil
from google.colab import drive

# Mount Google Drive (if not already mounted)
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# Define base project directory in Google Drive
base_dir = "/content/drive/MyDrive/Enhancing_EHRs_with_GenAI"

# Define folder structure
folders = [
    "data/raw",
    "data/images_processed",
    "data/ehr_notes_processed",
    "notebooks",
    "scripts",
    "docs"
]

# Define files to create (with minimal content)
files = {
    "data/mapping.csv": "file_id,image_path,note_path,diagnosis,icd10\n",
    "notebooks/01_data_prep.ipynb": '{"cells": [], "metadata": {}, "nbformat": 4, "nbformat_minor": 5}',
    "scripts/convert_dicoms.py": "# Script to convert DICOM files to PNG/JPG\n",
    "docs/dataset_sources.md": "# Dataset Sources\n- Brain Tumors Dataset: https://www.kaggle.com/datasets/mohammadhossein77/brain-tumors-dataset\n- EHR Dataset: https://www.kaggle.com/datasets/gauravsrivastav2507/ehr-dataset\n",
    "docs/cleaning_steps.md": "# Data Cleaning Steps\n",
    "docs/challenges.md": "# Challenges\n",
    "README.md": "# Enhancing EHRs with GenAI\nProject repository for preprocessing EHR and image data.\n"
}

# Create directories
print("Creating folder structure...")
for folder in folders:
    folder_path = os.path.join(base_dir, folder)
    os.makedirs(folder_path, exist_ok=True)
    print(f"Created: {folder_path}")

# Create files
print("\nCreating files...")
for file_path, content in files.items():
    full_path = os.path.join(base_dir, file_path)
    with open(full_path, "w") as f:
        f.write(content)
    print(f"Created: {full_path}")

# Move or copy datasets to data/raw/
raw_data_dir = os.path.join(base_dir, "data/raw")
print("\nMoving datasets to data/raw/...")
# Brain Tumors Dataset
brain_tumor_src = "/content/brain_tumor_data"
if os.path.exists(brain_tumor_src):
    brain_tumor_dst = os.path.join(raw_data_dir, "brain_tumor_data")
    shutil.copytree(brain_tumor_src, brain_tumor_dst, dirs_exist_ok=True)
    print(f"Copied Brain Tumors Dataset to {brain_tumor_dst}")
else:
    print(f"Brain Tumors Dataset not found at {brain_tumor_src}. Please ensure it's extracted.")

# EHR Dataset
ehr_src = "/content/ehr_data"
if os.path.exists(ehr_src):
    ehr_dst = os.path.join(raw_data_dir, "ehr_data")
    shutil.copytree(ehr_src, ehr_dst, dirs_exist_ok=True)
    print(f"Copied EHR Dataset to {ehr_dst}")
else:
    print(f"EHR Dataset not found at {ehr_src}. Please ensure it's extracted.")

# Verify folder structure
print("\nVerifying folder structure:")
!ls -R {base_dir}

Creating folder structure...
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/raw
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/notebooks
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/scripts
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/docs

Creating files...
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/mapping.csv
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/notebooks/01_data_prep.ipynb
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/scripts/convert_dicoms.py
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/docs/dataset_sources.md
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/docs/cleaning_steps.md
Created: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/docs/challenges.md
Created: /content/drive/MyDrive/Enhanci

In [4]:
# Install required packages
!pip install pillow opencv-python numpy

import os
from PIL import Image
import numpy as np
from google.colab import drive

# Mount Google Drive (if not already mounted)
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# Define paths based on your folder structure
base_dir = "/content/drive/MyDrive/Enhancing_EHRs_with_GenAI"
src_dir = os.path.join(base_dir, "data/raw/brain_tumor_data/Data")  # Source: Brain Tumors Dataset
dst_dir = os.path.join(base_dir, "data/images_processed")  # Destination: Processed images
os.makedirs(dst_dir, exist_ok=True)  # Create destination directory if it doesn't exist
target_size = (256, 256)  # Target image size

# Function to process and standardize images
def process_image(src_path, dst_path, target_size):
    try:
        # Open and convert image to grayscale
        img = Image.open(src_path).convert('L')
        # Resize to target size
        img = img.resize(target_size)
        # Save as PNG
        img.save(dst_path)
        print(f"Processed: {dst_path}")
    except Exception as e:
        print(f"Error processing {src_path}: {e}")

# Process all .jpg files recursively
print(f"Processing images from {src_dir}...")
image_count = 0
for root, _, files in os.walk(src_dir):
    for fn in files:
        if fn.lower().endswith(('.jpg', '.jpeg', '.png')):  # Handle common image formats
            src_path = os.path.join(root, fn)
            # Create output filename (preserve subfolder structure)
            rel_path = os.path.relpath(src_path, src_dir)
            outname = os.path.splitext(rel_path)[0] + '.png'
            dst_path = os.path.join(dst_dir, outname)
            # Ensure destination subfolder exists
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            # Process the image
            process_image(src_path, dst_path, target_size)
            image_count += 1

print(f"\nProcessed {image_count} images.")
print(f"Output saved to: {dst_dir}")

# Verify processed images
print("\nVerifying processed images:")
!ls -R {dst_dir}

Streaming output truncated to the last 5000 lines.
Processed: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_400_RO_.png
Processed: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_29_SP_.png
Processed: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_256_RO_.png
Processed: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_254_DA_.png
Processed: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_358_SP_.png
Processed: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_108.png
Processed: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_35.png
Processed: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_56_DA_.png
Processed: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_101_VF_.png
Processed: /content/drive/MyDrive

In [15]:
# Install required packages
!pip install pandas

import pandas as pd
import os
import re
from google.colab import drive

# Mount Google Drive (if not already mounted)
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# Define paths
base_dir = "/content/drive/MyDrive/Enhancing_EHRs_with_GenAI"
ehr_csv = "/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/raw/ehr_data/cancer_diagnosis_data.csv"
notes_dir = os.path.join(base_dir, "data/ehr_notes_processed")
os.makedirs(notes_dir, exist_ok=True)

# Step 1: Verify EHR CSV
print("Verifying EHR CSV...")
if not os.path.exists(ehr_csv):
    print(f"Error: CSV not found at {ehr_csv}")
    print(f"Files in {os.path.dirname(ehr_csv)}:", os.listdir(os.path.dirname(ehr_csv)))
    print("Re-downloading EHR Dataset...")
    !kaggle datasets download -d gauravsrivastav2507/ehr-dataset --force
    !unzip -o /content/ehr-dataset.zip -d /content/ehr_data
    !mkdir -p /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/raw/ehr_data
    !cp /content/ehr_data/*.csv /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/raw/ehr_data/
    print("Files in ehr_data:", os.listdir("/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/raw/ehr_data"))
    csv_files = glob.glob("/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/raw/ehr_data/*.csv")
    if csv_files:
        ehr_csv = csv_files[0]
        print(f"Updated CSV path: {ehr_csv}")
    else:
        raise FileNotFoundError("No CSV found in ehr_data.")

# Step 2: Load and analyze EHR Dataset
print("\nLoading EHR dataset...")
ehr_df = pd.read_csv(ehr_csv)
print("Shape:", ehr_df.shape)
print("Columns:", ehr_df.columns.tolist())
print("Sample data (first 5 rows):\n")
display(ehr_df.head())

# Analyze columns for text content
print("\nDetailed column analysis:")
text_column = None
for col in ehr_df.columns:
    dtype = ehr_df[col].dtype
    non_null = ehr_df[col].notna().sum()
    if dtype == "object":
        avg_len = ehr_df[col].str.len().mean() if non_null > 0 else 0
        sample_values = ehr_df[col].dropna().head(3).tolist()
        print(f"Column '{col}': dtype={dtype}, non-null={non_null}, avg_length={avg_len:.2f}, sample={sample_values}")
        if avg_len > 10 and non_null > 0:  # Relaxed threshold to detect text
            text_column = col
    else:
        print(f"Column '{col}': dtype={dtype}, non-null={non_null}")

# Manual override for text column (uncomment and set if known)
# text_column = "notes"  # Replace with actual column name, e.g., 'clinical_notes'

if not text_column:
    print("\nError: No suitable text column found (avg length > 10, non-null values).")
    print("Please specify a text column manually or confirm the dataset lacks notes.")
else:
    print(f"\nUsing column '{text_column}' for text extraction.")

    # Clean text function
    def clean_text(text):
        if pd.isna(text):
            return ""
        text = str(text).replace('\r', '\n')
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        return text

    # Extract and save text notes
    print(f"\nExtracting text notes to {notes_dir}...")
    note_count = 0
    for idx, row in ehr_df.iterrows():
        text = clean_text(row[text_column])
        if text:
            outfn = os.path.join(notes_dir, f"note_{idx}.txt")
            with open(outfn, 'w', encoding='utf-8') as f:
                f.write(text)
            print(f"Saved: {outfn}")
            note_count += 1
    print(f"\nProcessed {note_count} notes.")
    if note_count == 0:
        print(f"Warning: No non-empty text found in '{text_column}'. Check for missing values:")
        print(f"Missing values: {ehr_df[text_column].isna().sum()}")
        print(f"Sample data:\n{ehr_df[text_column].head(10)}")

# Verify notes
notes = sorted(glob.glob(os.path.join(notes_dir, "*.txt")))
print(f"\nFound {len(notes)} notes in {notes_dir}")
if notes:
    print("Sample note files:", notes[:5])
else:
    print("No notes generated. Proceed to mapping without notes.")

Verifying EHR CSV...

Loading EHR dataset...
Shape: (20000, 9)
Columns: ['Patient_ID', 'Age', 'Gender', 'Tumor_Size(cm)', 'Tumor_Type', 'Biopsy_Result', 'Treatment', 'Response_to_Treatment', 'Survival_Status']
Sample data (first 5 rows):



,Patient_ID,Age,Gender,Tumor_Size(cm),Tumor_Type,Biopsy_Result,Treatment,Response_to_Treatment,Survival_Status
0,c044501a-43ca-4a0c-8b8b-991439ba1b6a,52,Female,5.08,Benign,Positive,Surgery,No Response,Survived
1,b8900c4c-1232-4084-9432-5d02eba74d20,32,Female,0.80,Benign,Negative,Surgery,Complete Response,Survived
2,3004e2bc-8037-49cb-a542-d5612b73beab,70,Female,9.56,Benign,Positive,Radiation Therapy,Complete Response,Deceased
3,1df86af7-6745-4dea-b127-cbc9915079fc,21,Female,3.07,Malignant,Negative,Surgery,Partial Response,Survived
4,128e00c3-72e3-4031-a7f4-1165d7199cce,62,Male,7.17,Malignant,Positive,Radiation Therapy,Complete Response,Deceased


Streaming output truncated to the last 5000 lines.
Saved: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_15005.txt
Saved: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_15006.txt
Saved: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_15007.txt
Saved: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_15008.txt
Saved: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_15009.txt
Saved: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_15010.txt
Saved: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_15011.txt
Saved: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_15012.txt
Saved: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_15013.txt
Saved: /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note

In [16]:
import pandas as pd
import os
import glob
from google.colab import drive

# Mount Google Drive (if not already mounted)
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# Define paths
base_dir = "/content/drive/MyDrive/Enhancing_EHRs_with_GenAI"
ehr_csv = "/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/raw/ehr_data/cancer_diagnosis_data.csv"
notes_dir = os.path.join(base_dir, "data/ehr_notes_processed")
images_dir = os.path.join(base_dir, "data/images_processed")
mapping_csv = os.path.join(base_dir, "data/mapping.csv")
lookup_csv = os.path.join(base_dir, "data/icd_lookup.csv")
os.makedirs(os.path.dirname(mapping_csv), exist_ok=True)

# Step 1: Load images and notes
images = sorted(glob.glob(os.path.join(images_dir, "**/*.png"), recursive=True))
notes = sorted(glob.glob(os.path.join(notes_dir, "*.txt")))
print(f"Found {len(images)} images in {images_dir}")
print(f"Found {len(notes)} notes in {notes_dir}")
if len(images) != len(notes) and notes:
    print(f"Warning: Number of images ({len(images)}) and notes ({len(notes)}) differ. Pairing up to the smaller count.")

# Step 2: Verify or create icd_lookup.csv
if not os.path.exists(lookup_csv):
    print(f"Warning: {lookup_csv} not found. Creating dummy lookup table.")
    lookup_df = pd.DataFrame({
        'condition_keyword': ['tumor', 'cancer', 'normal'],
        'icd10_code': ['C71.9', 'C80.1', 'Z00.0']
    })
    lookup_df.to_csv(lookup_csv, index=False)
    print(f"Created dummy {lookup_csv}.")
else:
    lookup_df = pd.read_csv(lookup_csv)
    print("ICD Lookup Columns:", lookup_df.columns.tolist())
    print("Sample lookup data:\n")
    display(lookup_df.head())

# Function to suggest ICD-10 code
def suggest_icd(note_text, lookup_df):
    if pd.isna(note_text) or not note_text:
        return "UNKNOWN"
    t = note_text.lower()
    for _, r in lookup_df.iterrows():
        if r['condition_keyword'] in t:
            return r['icd10_code']
    return "UNKNOWN"

# Step 3: Load EHR dataset for diagnosis and text (if no notes)
diagnosis_column = None
icd10_column = None
text_column = None
ehr_df = None
if os.path.exists(ehr_csv):
    ehr_df = pd.read_csv(ehr_csv)
    print("EHR Dataset Columns:", ehr_df.columns.tolist())
    possible_diagnosis_columns = [col for col in ehr_df.columns if "diagnosis" in col.lower()]
    possible_icd10_columns = [col for col in ehr_df.columns if "icd10" in col.lower() or "icd" in col.lower()]
    if possible_diagnosis_columns:
        diagnosis_column = possible_diagnosis_columns[0]
        print(f"Using '{diagnosis_column}' for diagnosis.")
    if possible_icd10_columns:
        icd10_column = possible_icd10_columns[0]
        print(f"Using '{icd10_column}' for ICD-10 (bypassing lookup).")
    # Check for text column if no notes
    if not notes:
        print("No notes found. Checking EHR CSV for text column...")
        for col in ehr_df.columns:
            if ehr_df[col].dtype == "object" and ehr_df[col].str.len().mean() > 10:
                text_column = col
                break
        if text_column:
            print(f"Using '{text_column}' from EHR CSV for ICD-10 assignment.")
        else:
            print("No suitable text column in EHR CSV.")
else:
    print(f"EHR CSV not found at {ehr_csv}. Inferring diagnosis from image subfolders.")

# Step 4: Create/Update mapping.csv
print("\nCreating/Updating mapping.csv...")
rows = []
if notes:
    # Pair images with notes
    for i, img in enumerate(images[:min(len(images), len(notes))]):
        pid = f"{i+1:04d}"
        note_path = notes[i]
        # Read note text for ICD-10
        with open(note_path, 'r', encoding='utf-8') as f:
            note_text = f.read()
        icd10 = suggest_icd(note_text, lookup_df) if not icd10_column else \
                ehr_df[icd10_column].iloc[i] if i < len(ehr_df) else "UNKNOWN"
        diagnosis = ehr_df[diagnosis_column].iloc[i] if diagnosis_column and i < len(ehr_df) else \
                    os.path.basename(os.path.dirname(img)) if os.path.basename(os.path.dirname(img)) in ["Normal", "Tumor"] else "Unknown"
        rows.append({
            'file_id': pid,
            'image_path': img,
            'note_path': note_path,
            'diagnosis': diagnosis,
            'icd10': icd10
        })
else:
    # No notes, use EHR CSV text or images only
    print("No notes available. Mapping images only...")
    if text_column and ehr_df is not None:
        for i, img in enumerate(images[:min(len(images), len(ehr_df))]):
            pid = f"{i+1:04d}"
            note_text = ehr_df[text_column].iloc[i] if i < len(ehr_df) else ""
            icd10 = suggest_icd(note_text, lookup_df) if not icd10_column else \
                    ehr_df[icd10_column].iloc[i] if i < len(ehr_df) else "UNKNOWN"
            diagnosis = ehr_df[diagnosis_column].iloc[i] if diagnosis_column and i < len(ehr_df) else \
                        os.path.basename(os.path.dirname(img)) if os.path.basename(os.path.dirname(img)) in ["Normal", "Tumor"] else "Unknown"
            rows.append({
                'file_id': pid,
                'image_path': img,
                'note_path': '',
                'diagnosis': diagnosis,
                'icd10': icd10
            })
    else:
        print("No text column in EHR CSV. Setting ICD-10 to 'UNKNOWN'.")
        for i, img in enumerate(images):
            pid = f"{i+1:04d}"
            diagnosis = ehr_df[diagnosis_column].iloc[i] if diagnosis_column and i < len(ehr_df) else \
                        os.path.basename(os.path.dirname(img)) if os.path.basename(os.path.dirname(img)) in ["Normal", "Tumor"] else "Unknown"
            icd10 = ehr_df[icd10_column].iloc[i] if icd10_column and i < len(ehr_df) else "UNKNOWN"
            rows.append({
                'file_id': pid,
                'image_path': img,
                'note_path': '',
                'diagnosis': diagnosis,
                'icd10': icd10
            })

# Save updated mapping.csv
mapping_df = pd.DataFrame(rows)
mapping_df.to_csv(mapping_csv, index=False)
print(f"\nSaved mapping to {mapping_csv}")
print("Sample of updated mapping.csv:\n")
display(mapping_df.head())
if len(images) > len(notes) and notes:
    print(f"Warning: {len(images) - len(notes)} images were not paired with notes.")
!head {mapping_csv}

Found 21672 images in /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed
Found 20000 notes in /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed
ICD Lookup Columns: ['condition_keyword', 'icd10_code']
Sample lookup data:



,condition_keyword,icd10_code
0,tumor,C71.9
1,cancer,C80.1
2,normal,Z00.0


EHR Dataset Columns: ['Patient_ID', 'Age', 'Gender', 'Tumor_Size(cm)', 'Tumor_Type', 'Biopsy_Result', 'Treatment', 'Response_to_Treatment', 'Survival_Status']

Creating/Updating mapping.csv...

Saved mapping to /content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/mapping.csv
Sample of updated mapping.csv:



,file_id,image_path,note_path,diagnosis,icd10
0,0001,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,Normal,UNKNOWN
1,0002,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,Normal,UNKNOWN
2,0003,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,Normal,UNKNOWN
3,0004,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,Normal,UNKNOWN
4,0005,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,Normal,UNKNOWN


file_id,image_path,note_path,diagnosis,icd10
0001,/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_1.png,/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_0.txt,Normal,UNKNOWN
0002,/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_10.png,/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_1.txt,Normal,UNKNOWN
0003,/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_100.png,/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_10.txt,Normal,UNKNOWN
0004,/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_100_BR_.png,/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_100.txt,Normal,UNKNOWN
0005,/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed/Normal/N_100_DA_.png,/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/note_100

In [17]:
# Install required packages
!pip install pandas

import pandas as pd
import os
import re
from google.colab import drive

# Mount Google Drive (if not already mounted)
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# Define paths
base_dir = "/content/drive/MyDrive/Enhancing_EHRs_with_GenAI"
mapping_csv = os.path.join(base_dir, "data/mapping.csv")
notes_dir = os.path.join(base_dir, "data/ehr_notes_processed")
images_dir = os.path.join(base_dir, "data/images_processed")

# Step 1: Load mapping.csv
print("Loading mapping.csv...")
if not os.path.exists(mapping_csv):
    print(f"Error: {mapping_csv} not found. Please run the mapping script first.")
    raise FileNotFoundError("mapping.csv missing.")
mapping_df = pd.read_csv(mapping_csv)
print("Shape:", mapping_df.shape)
print("Columns:", mapping_df.columns.tolist())
print("Sample data (first 5 rows):\n")
display(mapping_df.head())

# Step 2: Integrity checks
print("\nRunning integrity checks...")

# Check for missing images
missing_images = mapping_df[~mapping_df['image_path'].apply(lambda x: os.path.exists(str(x)) if pd.notna(x) else False)]
print(f"Missing images: {len(missing_images)}")
if len(missing_images) > 0:
    print("Missing image paths (first 5):\n")
    display(missing_images[['file_id', 'image_path']].head())
    print("Suggestion: Verify that images exist in", images_dir)

# Check for missing notes (skip empty note_path)
missing_notes = mapping_df[
    (mapping_df['note_path'].notna()) &
    (mapping_df['note_path'] != '') &
    (~mapping_df['note_path'].apply(lambda x: os.path.exists(str(x))))
]
print(f"Missing notes: {len(missing_notes)}")
if len(missing_notes) > 0:
    print("Missing note paths (first 5):\n")
    display(missing_notes[['file_id', 'note_path']].head())
    print("Suggestion: Check if notes were extracted to", notes_dir)
else:
    empty_notes = mapping_df[mapping_df['note_path'].isna() | (mapping_df['note_path'] == '')]
    print(f"Empty note_path entries: {len(empty_notes)}")
    if len(empty_notes) == len(mapping_df):
        print("Warning: All note_path entries are empty. Dataset may lack notes. Run text extraction script to confirm.")

# Step 3: Sanity checks
print("\nRunning sanity checks...")

# Check for duplicate file_id
duplicate_ids = mapping_df[mapping_df['file_id'].duplicated(keep=False)]
print(f"Duplicate file_id entries: {len(duplicate_ids)}")
if len(duplicate_ids) > 0:
    print("Duplicate file_id entries (first 5):\n")
    display(duplicate_ids[['file_id', 'image_path', 'note_path']].head())
    print("Suggestion: Ensure file_id is unique for each row.")

# Check for missing or invalid diagnoses
missing_diagnoses = mapping_df[mapping_df['diagnosis'].isna() | (mapping_df['diagnosis'] == '') | (mapping_df['diagnosis'] == 'Unknown')]
print(f"Missing or 'Unknown' diagnoses: {len(missing_diagnoses)}")
if len(missing_diagnoses) > 0:
    print("Missing/Unknown diagnosis entries (first 5):\n")
    display(missing_diagnoses[['file_id', 'diagnosis']].head())
    print("Suggestion: Check EHR CSV for diagnosis column or verify image subfolders (Normal/Tumor).")

# Check ICD-10 format
# Updated regex: Matches A00-Z99 with optional decimal and up to 4 additional characters (e.g., C71.9, Z00.0, A123)
pattern = re.compile(r'^[A-Z][0-9]{2}(?:\.[0-9A-Za-z]{0,4})?$')
bad_icd = mapping_df[
    (~mapping_df['icd10'].isna()) &
    (mapping_df['icd10'] != '') &
    (mapping_df['icd10'] != 'UNKNOWN') &
    (~mapping_df['icd10'].apply(lambda x: bool(pattern.match(str(x)))))
]
print(f"Bad ICD-10 codes: {len(bad_icd)}")
if len(bad_icd) > 0:
    print("Bad ICD-10 codes (first 5):\n")
    display(bad_icd[['file_id', 'icd10']].head())
    print("Suggestion: Verify icd_lookup.csv or EHR CSV ICD-10 column for valid codes.")

# Step 4: Additional checks
print("\nAdditional consistency checks...")

# Check for orphaned notes (notes not referenced in mapping.csv)
notes = sorted(glob.glob(os.path.join(notes_dir, "*.txt")))
mapped_notes = set(mapping_df['note_path'].dropna())
orphaned_notes = [note for note in notes if note not in mapped_notes]
print(f"Orphaned notes (not in mapping.csv): {len(orphaned_notes)}")
if orphaned_notes:
    print("Orphaned note files (first 5):", orphaned_notes[:5])
    print("Suggestion: Ensure all notes are paired in mapping.csv.")

# Check image path format (should be in images_processed/Normal or Tumor)
invalid_image_paths = mapping_df[
    ~mapping_df['image_path'].str.contains('Normal|Tumor', case=False, na=False)
]
print(f"Image paths not in Normal/Tumor subfolders: {len(invalid_image_paths)}")
if len(invalid_image_paths) > 0:
    print("Invalid image paths (first 5):\n")
    display(invalid_image_paths[['file_id', 'image_path']].head())
    print("Suggestion: Ensure images are in .../images_processed/Normal/ or .../images_processed/Tumor/.")

# Summary
print("\nSummary of issues:")
print(f"- Missing images: {len(missing_images)}")
print(f"- Missing notes: {len(missing_notes)}")
print(f"- Empty note_path entries: {len(empty_notes)}")
print(f"- Duplicate file_id: {len(duplicate_ids)}")
print(f"- Missing/Unknown diagnoses: {len(missing_diagnoses)}")
print(f"- Bad ICD-10 codes: {len(bad_icd)}")
print(f"- Orphaned notes: {len(orphaned_notes)}")
print(f"- Invalid image paths: {len(invalid_image_paths)}")
if len(missing_images) == 0 and len(missing_notes) == 0 and len(duplicate_ids) == 0 and \
   len(bad_icd) == 0 and len(orphaned_notes) == 0 and len(invalid_image_paths) == 0:
    print("All integrity and sanity checks passed (except possibly empty note_path or missing diagnoses).")
else:
    print("Issues detected. Review suggestions above and run text extraction/mapping scripts as needed.")

Loading mapping.csv...
Shape: (20000, 5)
Columns: ['file_id', 'image_path', 'note_path', 'diagnosis', 'icd10']
Sample data (first 5 rows):



,file_id,image_path,note_path,diagnosis,icd10
0,1,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,Normal,UNKNOWN
1,2,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,Normal,UNKNOWN
2,3,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,Normal,UNKNOWN
3,4,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,Normal,UNKNOWN
4,5,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,/content/drive/MyDrive/Enhancing_EHRs_with_Gen...,Normal,UNKNOWN



Running integrity checks...
Missing images: 0
Missing notes: 0
Empty note_path entries: 0

Running sanity checks...
Duplicate file_id entries: 0
Missing or 'Unknown' diagnoses: 16934
Missing/Unknown diagnosis entries (first 5):



,file_id,diagnosis
3066,3067,Unknown
3067,3068,Unknown
3068,3069,Unknown
3069,3070,Unknown
3070,3071,Unknown


Suggestion: Check EHR CSV for diagnosis column or verify image subfolders (Normal/Tumor).
Bad ICD-10 codes: 0

Additional consistency checks...
Orphaned notes (not in mapping.csv): 0
Image paths not in Normal/Tumor subfolders: 0

Summary of issues:
- Missing images: 0
- Missing notes: 0
- Empty note_path entries: 0
- Duplicate file_id: 0
- Missing/Unknown diagnoses: 16934
- Bad ICD-10 codes: 0
- Orphaned notes: 0
- Invalid image paths: 0
All integrity and sanity checks passed (except possibly empty note_path or missing diagnoses).
